<a href="https://colab.research.google.com/github/uppikaur/aml/blob/master/CNNAssignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

In [ ]:
"""
Load Dataset
"""
img_size = (150, 150)
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    "cats_and_dogs_filtered/train",
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "cats_and_dogs_filtered/validation",
    image_size=img_size,
    batch_size=batch_size
)

In [ ]:
"""
Normalize Images
"""
normalization_layer = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

In [ ]:
"""
Basic CNN design
"""
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(150,150,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

In [ ]:
"""
Compile, Train, Evaluate
"""
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

In [ ]:
"""
Accuracy Plot
"""
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.legend(['Train', 'Validation'])
plt.title("CNN Accuracy")
plt.show()

In [ ]:
"""
Load pretrained model
Model 1: VGG16
         MobileNetV2
"""
base_model = tf.keras.applications.VGG16(
    input_shape=(150,150,3),
    include_top=False,
    weights='imagenet'
)

In [ ]:
"""
Freeze Base Layers
"""
for layer in base_model.layers:
    layer.trainable = False

In [ ]:
"""
Add Custom Layers
"""
x = base_model.output
x = layers.Flatten()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dense(1, activation='sigmoid')(x)

model_vgg = tf.keras.Model(inputs=base_model.input, outputs=x)

In [ ]:
"""
Compile & Train
"""

model_vgg.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_vgg = model_vgg.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

In [ ]:
"""
Transfer Learning Model 2: MobileNetV2
"""
base_model2 = tf.keras.applications.MobileNetV2(
    input_shape=(150,150,3),
    include_top=False,
    weights='imagenet'
)

base_model2.trainable = False

x = base_model2.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dense(1, activation='sigmoid')(x)

model_mobilenet = tf.keras.Model(inputs=base_model2.input, outputs=x)

model_mobilenet.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_mob = model_mobilenet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)